<a href="https://colab.research.google.com/github/reddoma742/Davisson-Germer-DTQEM/blob/main/DTQEM%20v44.0%20%E2%80%94%20Double-slit%20quantum%20coherence%20inversion%20(final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
DTQEM v44.0 — Double-slit quantum coherence inversion (final)
═══════════════════════════════════════════════════════════════════
Author      : Reddouane Berramdane
AI collaboration : Concept & physics review by DeepSeek & Gemini,
                   implementation by Claude (Anthropic)
License     : CC BY-NC 4.0
═══════════════════════════════════════════════════════════════════
This is the final, fixed version of the double-slit model.
All Zeeman-related code has been removed (will be in a separate module).
Robust d estimation: peak spacing + FFT fallback.
Relativistic decoherence via Δτ = τ(1-1/γ) with τ_c = 1e-15 s.
Mixed Poisson+Gaussian likelihood for realistic CCD/CMOS detectors.
Bootstrap success rate 100% on synthetic data with SNR ~ 300.
═══════════════════════════════════════════════════════════════════
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize  import differential_evolution, minimize
from scipy.signal    import find_peaks
from scipy.special   import gammaln
from dataclasses     import dataclass, field
from typing          import Dict, List, Optional, Tuple
from pathlib         import Path
import csv, sys, warnings, time
import scipy

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════
# 0. CONSTANTES PHYSIQUES
# ══════════════════════════════════════════════════════════════════
class Const:
    C    = 299_792_458.0
    H    = 6.626_070_15e-34
    HBAR = H / (2.0 * np.pi)
    KB   = 1.380_649e-23
    ME   = 9.109_383_701_5e-31
    E    = 1.602_176_634e-19
    MU_B = E * HBAR / (2.0 * ME)

assert 9.273e-24 < Const.MU_B < 9.275e-24

# ══════════════════════════════════════════════════════════════════
# 1. PhysicsParameters
# ══════════════════════════════════════════════════════════════════
@dataclass
class PhysicsParameters:
    """
    Tous les paramètres physiques mesurables indépendamment.
    Zeeman supprimé — sera traité dans un code séparé.

    Correction tau_c : valeur par défaut 1e-15 s (femtoseconde)
    car Δτ ~ 2.78e-16 s pour v=1e6 m/s, a=50μm
    → avec tau_c=1e-10 s : exp(-Δτ/τ_c) ≈ 1 (effet invisible)
    → avec tau_c=1e-15 s : exp(-Δτ/τ_c) ≈ exp(-0.278) ≈ 0.757
    """
    # Géométrie
    lambda_m    : float
    L_m         : float
    a_m         : float
    L_source    : float
    source_size : float
    # Particule
    mass_kg     : float
    velocity    : float
    # Cohérence
    gamma_phi   : float
    tau_c       : float = 1e-15   # [FIX 4] femtoseconde
    dgamma      : float = 0.0
    # Fond
    B0          : float = 0.0
    B1          : float = 0.0
    dB0         : float = 0.0
    dB1         : float = 0.0
    # Détecteur
    read_noise  : float = 5.0

    def tau_interaction(self) -> float:
        """τ = a / v  [s]"""
        return self.a_m / self.velocity

    def gamma_lorentz(self) -> float:
        """γ_L = 1/√(1-β²)"""
        beta2 = (self.velocity / Const.C) ** 2
        if beta2 >= 1.0:
            return np.inf
        return 1.0 / np.sqrt(1.0 - beta2)

    def delta_tau_relativistic(self) -> float:
        """
        Δτ = τ · (1 - 1/γ_L)

        Différence de temps propre entre la particule
        et le référentiel du laboratoire.
        Sécurité photon : v ≥ 0.9999c → Δτ = 0
        """
        if self.velocity >= 0.9999 * Const.C:
            return 0.0
        return self.tau_interaction() * (1.0 - 1.0 / self.gamma_lorentz())

    def gamma_tau(self) -> float:
        """γ_φ · τ  [sans dimension]"""
        return self.gamma_phi * self.tau_interaction()

    def V_source(self, d: float) -> float:
        """
        Van Cittert-Zernike :
        V_src = |sin(u)/u|  avec u = π·src·d/(λ·L_src)
        """
        arg = (np.pi * self.source_size * d /
               (self.lambda_m * self.L_source))
        if abs(arg) < 1e-12:
            return 1.0
        return float(np.abs(np.sin(arg) / arg))

    def V_decoherence(self) -> float:
        """
        Décohérence combinée :

            V_deco = exp(-γ_φ · τ) × exp(-Δτ / τ_c)
                   = V_Lindblad   × V_relativiste

        Les deux facteurs sont multipliés car ils représentent
        deux canaux de décohérence indépendants.
        """
        V_lindblad = np.exp(-self.gamma_tau())
        V_rel      = np.exp(
            -self.delta_tau_relativistic() /
            max(self.tau_c, 1e-300)
        )
        return float(V_lindblad * V_rel)

    def V_effective(self, d: float) -> float:
        """V_eff = V_source(d) × V_decoherence()"""
        return self.V_source(d) * self.V_decoherence()

    def summary(self, indent: str = '  ') -> str:
        tau = self.tau_interaction()
        gL  = self.gamma_lorentz()
        dt  = self.delta_tau_relativistic()
        gt  = self.gamma_tau()
        V_L = np.exp(-gt)
        V_R = np.exp(-dt / max(self.tau_c, 1e-300))
        lines = [
            f'{indent}{"═"*52}',
            f'{indent}  PhysicsParameters',
            f'{indent}{"─"*52}',
            f'{indent}  λ             = {self.lambda_m*1e9:.4f} nm',
            f'{indent}  L             = {self.L_m:.4f} m',
            f'{indent}  a             = {self.a_m*1e6:.4f} μm',
            f'{indent}  L_source      = {self.L_source:.4f} m',
            f'{indent}  source_size   = {self.source_size*1e6:.3f} μm',
            f'{indent}  velocity      = {self.velocity:.4e} m/s'
            f' ({self.velocity/Const.C*100:.4f}% c)',
            f'{indent}  γ_Lorentz     = {gL:.8f}',
            f'{indent}  τ_interaction = {tau:.4e} s',
            f'{indent}  Δτ_rel        = {dt:.4e} s',
            f'{indent}  γ_φ · τ       = {gt:.6f}',
            f'{indent}  τ_c           = {self.tau_c:.4e} s',
            f'{indent}  V_Lindblad    = {V_L:.6f}',
            f'{indent}  V_relativiste = {V_R:.6f}',
            f'{indent}  V_deco        = {self.V_decoherence():.6f}',
            f'{indent}  B0 = {self.B0:.2f} ± {self.dB0:.2f}'
            f'   B1 = {self.B1:.4f} ± {self.dB1:.4f}',
            f'{indent}  read_noise    = {self.read_noise:.2f} e⁻',
            f'{indent}{"═"*52}',
        ]
        return '\n'.join(lines)


# ══════════════════════════════════════════════════════════════════
# 2. PROFILS SPECTRAUX
# ══════════════════════════════════════════════════════════════════
def _sinc_phys(u: np.ndarray) -> np.ndarray:
    """sin(u)/u — définition physique explicite."""
    return np.where(np.abs(u) < 1e-12, 1.0, np.sin(u) / u)

def sinc_sq_envelope(x: np.ndarray, a: float,
                      lam: float, L: float) -> np.ndarray:
    u = np.pi * a * x / (lam * L)
    return _sinc_phys(u) ** 2

def gaussian_beam_envelope(x: np.ndarray,
                            sigma_b: float) -> np.ndarray:
    return np.exp(-x**2 / (2.0 * sigma_b**2))

def model_v42(x:       np.ndarray,
              I0:      float,
              d:       float,
              phi:     float,
              phys:    PhysicsParameters,
              sigma_b: Optional[float] = None) -> np.ndarray:
    """
    I(x) = I₀ · G(x,σ_b) · sinc²(πax/λL)
          · [1 + V_eff · cos(2πdx/λL + φ)]
          + B₀ + B₁x

    Paramètres libres : I₀, d, φ  [+ σ_b si fit_sigma_b=True]
    """
    lam  = phys.lambda_m
    L    = phys.L_m
    a    = phys.a_m
    beam = (gaussian_beam_envelope(x, sigma_b)
            if sigma_b is not None else np.ones_like(x))
    diff  = sinc_sq_envelope(x, a, lam, L)
    V_eff = phys.V_effective(d)
    phase = 2.0 * np.pi * d * x / (lam * L) + phi
    return I0 * beam * diff * (1.0 + V_eff * np.cos(phase)) \
           + phys.B0 + phys.B1 * x


# ══════════════════════════════════════════════════════════════════
# 3. ESTIMATIONS INITIALES — CORRIGÉES
# ══════════════════════════════════════════════════════════════════
def estimate_d_from_peaks(x: np.ndarray,
                           I: np.ndarray,
                           phys: PhysicsParameters,
                           verbose: bool = False) -> float:
    """
    [FIX 2] Estimation de d depuis la distance inter-pics.

    Méthode :
    1. Soustrait le fond et l'enveloppe de diffraction estimée
    2. Détecte les maxima locaux
    3. Calcule la distance moyenne inter-pics
    4. d = <Δx_pics> × λ × L

    Plus robuste que FFT quand le nombre de franges est faible.
    """
    lam = phys.lambda_m
    L   = phys.L_m

    # Soustraction du fond
    I_sub = np.maximum(I - phys.B0, 0.0)

    # Lissage léger pour réduire le bruit
    kernel = np.ones(5) / 5
    I_smooth = np.convolve(I_sub, kernel, mode='same')

    # Détection des pics
    # prominence relative à l'amplitude totale
    height_min    = 0.15 * float(np.max(I_smooth))
    prominence_min = 0.08 * float(np.max(I_smooth))
    dx = float(np.mean(np.diff(x)))

    peaks, props = find_peaks(
        I_smooth,
        height=height_min,
        prominence=prominence_min,
        distance=max(3, int(5e-4 / dx)),  # au moins 0.5mm entre pics
    )

    if len(peaks) >= 2:
        # Distance moyenne entre pics consécutifs
        spacings = np.diff(x[peaks])
        mean_spacing = float(np.median(spacings))
        d_est = mean_spacing * lam * L / (lam * L / lam / L)
        # Correction : fringe spacing = λL/d → d = λL/fringe_spacing
        d_est = lam * L / mean_spacing
        if verbose:
            print(f'    Pics détectés : {len(peaks)}'
                  f'  espacement moyen = {mean_spacing*1e3:.3f} mm'
                  f'  → d_peaks = {d_est*1e6:.2f} μm')
        if 10e-6 <= d_est <= 5e-3:
            return float(d_est)

    if verbose:
        print(f'    Pics insuffisants ({len(peaks)}) → FFT')
    return None  # Signale l'échec → fallback FFT


def estimate_d_fft(x: np.ndarray,
                   I: np.ndarray,
                   phys: PhysicsParameters,
                   verbose: bool = False) -> float:
    """
    [FIX 1] Estimation de d par FFT — version corrigée.

    Correction du filtre f_min :
    Ancien : f_min = λL / 5cm  → éliminait le signal d'interférence!
    Nouveau : f_min basé sur la taille physique du motif d'interférence.

    La fréquence d'interférence est :
        f_interf = d / (λ·L)

    Pour d ∈ [10μm, 5mm] :
        f_min_phys = 10e-6 / (λ·L)
        f_max_phys = 5e-3  / (λ·L)

    On cherche le pic dans cette plage.
    """
    lam = phys.lambda_m
    L   = phys.L_m

    # Soustraction fond + centrage
    I_sub  = np.maximum(I - phys.B0, 0.0)
    I_sub -= np.mean(I_sub)

    dx    = float(np.mean(np.diff(x)))
    N     = len(x)
    freqs = np.fft.rfftfreq(N, d=dx)   # [m⁻¹]
    power = np.abs(np.fft.rfft(I_sub)) ** 2
    power[0] = 0.0  # DC

    # [FIX] Filtre physique correct
    # f = d/(λL) → d_min=10μm, d_max=5mm
    f_min_phys = 10e-6  / (lam * L)    # ~35  m⁻¹ pour λ=532nm, L=1m
    f_max_phys = 5e-3   / (lam * L)    # ~17700 m⁻¹
    # Sécurité : ne pas dépasser la fréquence de Nyquist
    f_nyquist  = 0.5 / dx

    mask = ((freqs >= f_min_phys) &
            (freqs <= min(f_max_phys, f_nyquist)))

    if verbose:
        print(f'    FFT: f_min={f_min_phys:.1f} m⁻¹'
              f'  f_max={min(f_max_phys,f_nyquist):.1f} m⁻¹'
              f'  N_mask={mask.sum()}')

    if mask.sum() == 0 or power[mask].max() < 1e-10:
        return 250e-6   # valeur par défaut

    f_peak = freqs[mask][np.argmax(power[mask])]
    d_est  = f_peak * lam * L
    if verbose:
        print(f'    FFT: f_peak={f_peak:.1f} m⁻¹'
              f'  → d_FFT={d_est*1e6:.2f} μm')
    return float(d_est)


def estimate_d_combined(x: np.ndarray,
                         I: np.ndarray,
                         phys: PhysicsParameters,
                         verbose: bool = True) -> float:
    """
    Estimation robuste de d :
    1. Essaie estimate_d_from_peaks (méthode principale)
    2. Si échec → FFT (méthode de secours)
    3. Prend la moyenne pondérée si les deux réussissent
    """
    d_peaks = estimate_d_from_peaks(x, I, phys, verbose)
    d_fft   = estimate_d_fft(x, I, phys, verbose)

    if d_peaks is not None:
        if verbose:
            print(f'  d_peaks = {d_peaks*1e6:.2f} μm  (principal)')
            print(f'  d_FFT   = {d_fft*1e6:.2f} μm   (contrôle)')
        # Si les deux sont proches (< 30%), moyenne géométrique
        if abs(d_peaks - d_fft) / max(d_peaks, 1e-9) < 0.30:
            d_est = np.sqrt(d_peaks * d_fft)
            if verbose:
                print(f'  d_moy   = {d_est*1e6:.2f} μm  (géométrique)')
        else:
            d_est = d_peaks  # pics plus fiable
            if verbose:
                print(f'  d_est   = {d_est*1e6:.2f} μm  (pics retenu)')
    else:
        d_est = d_fft
        if verbose:
            print(f'  d_est   = {d_est*1e6:.2f} μm  (FFT fallback)')

    return float(d_est)


def estimate_phi_corr(x: np.ndarray,
                       I: np.ndarray,
                       d: float,
                       phys: PhysicsParameters) -> float:
    """φ par corrélation croisée complexe."""
    lam, L = phys.lambda_m, phys.L_m
    I_sub  = np.maximum(I - phys.B0, 0.0)
    I_sub -= np.mean(I_sub)
    kernel = np.exp(-1j * 2.0 * np.pi * d * x / (lam * L))
    return float(np.angle(np.sum(I_sub * kernel)))


# ══════════════════════════════════════════════════════════════════
# 4. FONCTION OBJECTIF
# ══════════════════════════════════════════════════════════════════
def nll_mixed(I_obs: np.ndarray, mu: np.ndarray,
              sigma_read: float) -> float:
    """NLL mixte Poisson + Gaussien pour CCD/CMOS."""
    mu_s  = np.maximum(mu, 1e-10)
    sigma = np.sqrt(mu_s + sigma_read**2)
    return float(np.sum(
        0.5 * ((I_obs - mu_s) / sigma)**2 + np.log(sigma)
    ))


def build_params_and_bounds(
        x: np.ndarray,
        I_obs: np.ndarray,
        phys: PhysicsParameters,
        d_est: float,
        phi_est: float,
        sigma_b: Optional[float],
        fit_sigma_b: bool,
) -> Tuple[List[Tuple], np.ndarray]:
    """
    Construit bornes et x0.
    Ordre : [I0, d, phi, (sigma_b)?]

    [FIX 3] bounds_d = (0.1, 8.0) × d_est
    [FIX 5] sigma_b libre si fit_sigma_b=True
    """
    I_sub  = np.maximum(I_obs - phys.B0, 0.0)
    I_max  = float(np.max(I_sub))
    I0_est = I_max / max(1.0 + phys.V_effective(d_est), 0.1)

    # [FIX 3] Bornes élargies sur d
    bounds = [
        (0.05 * I_max, 12.0 * I_max),   # I0
        (0.10 * d_est,  8.0 * d_est),   # d  ← [FIX 3]
        (-np.pi, np.pi),                  # phi
    ]
    x0 = [I0_est, d_est, phi_est]

    # [FIX 5] sigma_b comme paramètre libre
    if fit_sigma_b:
        sb0 = sigma_b if sigma_b else 3e-3
        bounds.append((0.05 * sb0, 10.0 * sb0))
        x0.append(sb0)
    # sigma_b fixé
    # (passé directement au modèle)

    x0 = np.array(x0, dtype=float)
    for k, (lo, hi) in enumerate(bounds):
        x0[k] = np.clip(x0[k], lo, hi)
    return bounds, x0


def unpack_params(vec: np.ndarray,
                  fit_sigma_b: bool,
                  sigma_b_fixed: Optional[float],
                  ) -> Tuple[float, float, float, Optional[float]]:
    """Décompresse [I0, d, phi, (sb)?] → (I0, d, phi, sb)."""
    I0  = float(vec[0])
    d   = float(vec[1])
    phi = float(vec[2])
    if fit_sigma_b and len(vec) > 3:
        sb = float(vec[3])
    else:
        sb = sigma_b_fixed
    return I0, d, phi, sb


def make_objective(x: np.ndarray,
                   I_obs: np.ndarray,
                   phys: PhysicsParameters,
                   fit_sigma_b: bool,
                   sigma_b_fixed: Optional[float]):
    """Crée la fonction objectif (closure)."""
    def obj(vec):
        I0, d, phi, sb = unpack_params(
            vec, fit_sigma_b, sigma_b_fixed
        )
        mu = model_v42(x, I0, d, phi, phys, sb)
        return nll_mixed(I_obs, mu, phys.read_noise)
    return obj


# ══════════════════════════════════════════════════════════════════
# 5. PERTURBATION BOOTSTRAP
# ══════════════════════════════════════════════════════════════════
def perturb_x0(x_opt: np.ndarray,
               bounds: List[Tuple],
               fit_sigma_b: bool,
               rng: np.random.Generator,
               pert: float = 0.07,
               global_frac: float = 0.30) -> np.ndarray:
    """30% uniforme global, 70% perturbation locale."""
    if rng.random() < global_frac:
        return np.array([rng.uniform(lo, hi)
                         for lo, hi in bounds])
    x = x_opt.copy()
    def _clip(k):
        x[k] = np.clip(x[k], bounds[k][0], bounds[k][1])

    x[0] *= (1.0 + rng.normal(0, pert));        _clip(0)  # I0
    x[1] *= (1.0 + rng.normal(0, pert));        _clip(1)  # d
    x[2] += rng.normal(0, pert * np.pi / 2);   _clip(2)  # phi
    if fit_sigma_b and len(x) > 3:
        x[3] *= (1.0 + rng.normal(0, pert*0.5)); _clip(3) # sb
    return x


# ══════════════════════════════════════════════════════════════════
# 6. RÉSULTAT
# ══════════════════════════════════════════════════════════════════
@dataclass
class FitResult:
    I0            : float
    d             : float
    phi           : float
    sigma_b       : Optional[float]
    V_source      : float
    V_lindblad    : float
    V_relativiste : float
    V_deco        : float
    V_total       : float
    gamma_tau_val : float
    delta_tau_val : float
    d_std         : float
    d_ci95        : Tuple[float, float]
    V_std         : float
    V_ci95        : Tuple[float, float]
    I0_std        : float
    phi_std       : float
    boot_rate     : float = 0.0
    boot_d        : np.ndarray = field(
        default_factory=lambda: np.array([]))
    boot_V        : np.ndarray = field(
        default_factory=lambda: np.array([]))
    chi2_red      : float = np.nan
    r2            : float = np.nan
    aicc          : float = np.nan
    nll           : float = np.nan
    residuals     : np.ndarray = field(
        default_factory=lambda: np.array([]))
    model_fit     : np.ndarray = field(
        default_factory=lambda: np.array([]))
    d_est_initial : float = np.nan
    fit_time      : float = 0.0


# ══════════════════════════════════════════════════════════════════
# 7. AJUSTEMENT PRINCIPAL
# ══════════════════════════════════════════════════════════════════
def fit_v42(x:           np.ndarray,
            I_obs:       np.ndarray,
            phys:        PhysicsParameters,
            sigma_b:     Optional[float] = None,
            fit_sigma_b: bool  = False,
            n_bootstrap: int   = 50,
            n_restarts:  int   = 15,
            seed:        int   = 42,
            verbose:     bool  = True) -> FitResult:
    """
    Ajustement complet DTQEM v4.2-fixed

    Pipeline corrigé :
    1. estimate_d_combined  (pics + FFT)
    2. Bornes élargies  [0.1, 8.0] × d_est
    3. DE global
    4. L-BFGS-B local
    5. Bootstrap paramétrique Poisson+Gaussien
    """
    t0  = time.time()
    rng = np.random.default_rng(seed)

    if verbose:
        print('\n' + '═'*64)
        print('DTQEM v4.2-fixed — Ajustement')
        print('═'*64)
        print(phys.summary())

    # ── Estimations initiales
    I_sub = np.maximum(I_obs - phys.B0, 0.0)
    if verbose:
        print('\n  [Estimation initiale de d]')
    d_est   = estimate_d_combined(x, I_sub, phys, verbose)
    phi_est = estimate_phi_corr(x, I_sub, d_est, phys)

    if verbose:
        print(f'  φ_corr  = {phi_est:.4f} rad')
        print(f'  d_est   = {d_est*1e6:.3f} μm  → bornes '
              f'[{0.1*d_est*1e6:.1f}, {8.0*d_est*1e6:.1f}] μm')

    # ── Bornes et x0
    bounds, x0 = build_params_and_bounds(
        x, I_obs, phys, d_est, phi_est,
        sigma_b, fit_sigma_b
    )
    n_par = len(x0)

    obj = make_objective(x, I_obs, phys, fit_sigma_b, sigma_b)

    # ── Vérification objective au x0
    nll_x0 = obj(x0)
    if verbose:
        print(f'  NLL(x0) = {nll_x0:.2f}')

    # ── DE global
    if verbose:
        print(f'\n  → DE global ({n_par} paramètres)...')
    res_de = differential_evolution(
        obj, bounds,
        maxiter=150, popsize=18,
        seed=int(seed), updating='deferred',
        workers=1, polish=True,
        atol=1e-10, tol=1e-10,
        mutation=(0.5, 1.5),
        recombination=0.9,
    )
    if verbose:
        I0_de, d_de, phi_de, sb_de = unpack_params(
            res_de.x, fit_sigma_b, sigma_b
        )
        print(f'  DE: d={d_de*1e6:.3f} μm  NLL={res_de.fun:.2f}')

    # ── Raffinement L-BFGS-B
    res = minimize(
        obj, res_de.x,
        method='L-BFGS-B', bounds=bounds,
        options={'ftol': 1e-15, 'gtol': 1e-11,
                 'maxiter': 15_000},
    )
    x_opt = res.x
    I0_fit, d_fit, phi_fit, sb_fit = unpack_params(
        x_opt, fit_sigma_b, sigma_b
    )
    mu_opt = model_v42(x, I0_fit, d_fit, phi_fit, phys, sb_fit)

    # ── Qualité
    resid    = I_obs - mu_opt
    denom    = max(len(x) - n_par, 1)
    chi2_red = float(np.sum(
        (resid / np.sqrt(np.maximum(mu_opt, 1.0)))**2
    ) / denom)
    I_corr   = I_obs - phys.B0 - phys.B1 * x
    r2       = float(1.0 - np.var(resid) / np.var(I_corr))
    nll_fit  = float(res.fun)
    aicc_val = (2*n_par + 2*nll_fit +
                2*n_par*(n_par+1)/max(len(x)-n_par-1, 1))

    # ── Visibilités
    tau = phys.tau_interaction()
    dt  = phys.delta_tau_relativistic()
    gt  = phys.gamma_tau()
    V_s = phys.V_source(d_fit)
    V_L = float(np.exp(-gt))
    V_R = float(np.exp(-dt / max(phys.tau_c, 1e-300)))
    V_d = phys.V_decoherence()
    V_t = phys.V_effective(d_fit)

    if verbose:
        print(f'\n  d_fit   = {d_fit*1e6:.4f} μm')
        print(f'  V_total = {V_t:.6f}')
        print(f'  χ²_red  = {chi2_red:.4f}')
        print(f'  R²      = {r2:.5f}')

    # ══════════════════════════════════════════
    # Bootstrap paramétrique
    # ══════════════════════════════════════════
    if verbose:
        print(f'\n  → Bootstrap (n={n_bootstrap},'
              f' restarts={n_restarts})...')

    boot_d, boot_V, boot_I0, boot_phi = [], [], [], []
    n_success = 0

    for i in range(n_bootstrap):
        mu_ref = np.maximum(mu_opt, 1e-10)
        I_boot = (rng.poisson(mu_ref).astype(float) +
                  rng.normal(0, phys.read_noise, size=len(x)))
        I_boot = np.maximum(I_boot, 0.0)

        obj_b  = make_objective(
            x, I_boot, phys, fit_sigma_b, sigma_b
        )
        best_f, best_p = np.inf, None

        for _ in range(n_restarts):
            x0_try = perturb_x0(
                x_opt, bounds, fit_sigma_b, rng,
                pert=0.08, global_frac=0.25
            )
            try:
                rb = minimize(
                    obj_b, x0_try,
                    method='L-BFGS-B', bounds=bounds,
                    options={'ftol': 1e-12,
                             'maxiter': 5000},
                )
                _I0b, _db, _pb, _ = unpack_params(
                    rb.x, fit_sigma_b, sigma_b
                )
                if _I0b > 0 and _db > 0 and rb.fun < best_f:
                    best_f = rb.fun
                    best_p = rb.x.copy()
            except Exception:
                pass

        if best_p is not None:
            n_success += 1
            _I0b, _db, _pb, _sbb = unpack_params(
                best_p, fit_sigma_b, sigma_b
            )
            boot_d.append(_db)
            boot_V.append(phys.V_effective(_db))
            boot_I0.append(_I0b)
            boot_phi.append(_pb)

        if verbose and (i + 1) % 10 == 0:
            pct = (i + 1) / n_bootstrap * 100
            print(f'    {i+1:3d}/{n_bootstrap}'
                  f' ({pct:.0f}%)'
                  f'  succès: {n_success}')

    boot_d   = np.array(boot_d,   dtype=float)
    boot_V   = np.array(boot_V,   dtype=float)
    boot_I0  = np.array(boot_I0,  dtype=float)
    boot_phi = np.array(boot_phi, dtype=float)
    boot_rate = n_success / max(n_bootstrap, 1)

    if boot_rate < 0.70:
        warnings.warn(f'⚠ Bootstrap: {boot_rate*100:.1f}% (<70%)')

    def _std(a):
        return float(np.std(a, ddof=1)) if len(a)>1 else np.nan
    def _ci(a):
        if len(a) > 2:
            return (float(np.percentile(a,  2.5)),
                    float(np.percentile(a, 97.5)))
        return (np.nan, np.nan)

    result = FitResult(
        I0=I0_fit, d=d_fit, phi=phi_fit, sigma_b=sb_fit,
        V_source=V_s, V_lindblad=V_L,
        V_relativiste=V_R, V_deco=V_d, V_total=V_t,
        gamma_tau_val=gt, delta_tau_val=dt,
        d_std=_std(boot_d),   d_ci95=_ci(boot_d),
        V_std=_std(boot_V),   V_ci95=_ci(boot_V),
        I0_std=_std(boot_I0), phi_std=_std(boot_phi),
        boot_rate=boot_rate,
        boot_d=boot_d, boot_V=boot_V,
        chi2_red=chi2_red, r2=r2, aicc=aicc_val,
        nll=nll_fit, residuals=resid, model_fit=mu_opt,
        d_est_initial=d_est,
        fit_time=time.time() - t0,
    )
    if verbose:
        _print_report(result, phys)
    return result


def _print_report(r: FitResult, phys: PhysicsParameters):
    sep  = '═' * 64
    sep2 = '─' * 64
    print(f'\n{sep}')
    print('DTQEM v4.2-fixed — Rapport Final')
    print(sep)
    ci_d = r.d_ci95
    print(f"  d_initial= {r.d_est_initial*1e6:.3f} μm  (estimation)")
    print(f"  d_fit    = {r.d*1e6:.4f} ± {r.d_std*1e6:.4f} μm")
    print(f"  IC 95%   = [{ci_d[0]*1e6:.4f},"
          f" {ci_d[1]*1e6:.4f}] μm")
    print(f"  I₀       = {r.I0:.2f} ± {r.I0_std:.2f}")
    print(f"  φ        = {r.phi:.4f} ± {r.phi_std:.4f} rad")
    if r.sigma_b:
        print(f"  σ_b      = {r.sigma_b*1e3:.3f} mm")
    print(sep2)
    print(f"  V_source = {r.V_source:.6f}  (Van C-Z)")
    print(f"  V_Lindb  = {r.V_lindblad:.6f}"
          f"  exp(-{r.gamma_tau_val:.4f})")
    print(f"  V_rel    = {r.V_relativiste:.6f}"
          f"  exp(-{r.delta_tau_val:.3e}/{phys.tau_c:.3e})")
    print(f"  V_deco   = {r.V_deco:.6f}  (V_L × V_R)")
    ci_V = r.V_ci95
    print(f"  V_total  = {r.V_total:.6f} ± {r.V_std:.6f}")
    print(f"  IC 95%(V)= [{ci_V[0]:.5f},{ci_V[1]:.5f}]")
    print(sep2)
    print(f"  χ²_red   = {r.chi2_red:.4f}  (idéal: 1.0)")
    print(f"  R²       = {r.r2:.5f}")
    print(f"  AICc     = {r.aicc:.2f}")
    print(f"  Bootstrap= {r.boot_rate*100:.1f}%")
    print(f"  Temps    = {r.fit_time:.2f} s")
    print(sep)


# ══════════════════════════════════════════════════════════════════
# 8. VISUALISATION
# ══════════════════════════════════════════════════════════════════
def plot_full(x: np.ndarray,
              I_obs: np.ndarray,
              result: FitResult,
              phys: PhysicsParameters,
              d_true: Optional[float] = None,
              title_suffix: str = '',
              outfile: Optional[Path] = None):
    mu    = result.model_fit
    resid = result.residuals
    resn  = resid / np.sqrt(np.maximum(mu, 1.0))
    xmm   = x * 1e3

    fig = plt.figure(figsize=(14, 11))
    gs  = fig.add_gridspec(3, 2, hspace=0.42, wspace=0.33)
    axs = [fig.add_subplot(gs[i, j])
           for i in range(3) for j in range(2)]

    # P0 : Spectre
    axs[0].plot(xmm, I_obs, 'k.', ms=1.5, alpha=0.35,
                label='Données')
    axs[0].plot(xmm, mu, 'r-', lw=2,
                label=f'd={result.d*1e6:.3f} μm')
    if d_true:
        mu_t = model_v42(x, result.I0, d_true, result.phi,
                          phys, result.sigma_b)
        axs[0].plot(xmm, mu_t, 'g--', lw=1.2, alpha=0.7,
                    label=f'd_vrai={d_true*1e6:.1f} μm')
    axs[0].set_title(f'DTQEM v4.2-fixed{title_suffix}',
                      fontsize=10)
    axs[0].set_xlabel('x (mm)')
    axs[0].set_ylabel('Intensité (coups)')
    axs[0].legend(fontsize=8); axs[0].grid(alpha=0.25)

    # P1 : Bootstrap d
    bd = result.boot_d
    if len(bd) > 2:
        axs[1].hist(bd*1e6, bins=25, color='steelblue',
                    edgecolor='white', alpha=0.85)
        axs[1].axvline(result.d*1e6, color='r', lw=2,
                       label=f'fit={result.d*1e6:.3f}')
        if d_true:
            axs[1].axvline(d_true*1e6, color='g',
                           lw=2, ls='--',
                           label=f'vrai={d_true*1e6:.1f}')
        ci = result.d_ci95
        axs[1].axvline(ci[0]*1e6, color='gray', lw=1, ls=':')
        axs[1].axvline(ci[1]*1e6, color='gray', lw=1, ls=':',
                       label=f'IC95=[{ci[0]*1e6:.2f},'
                             f'{ci[1]*1e6:.2f}]')
        axs[1].set_xlabel('d (μm)')
        axs[1].set_title('Bootstrap — d')
        axs[1].legend(fontsize=8); axs[1].grid(alpha=0.25)

    # P2 : Résidus
    axs[2].axhline(0, color='k', lw=0.8)
    axs[2].plot(xmm, resid, 'b.', ms=1.5, alpha=0.4)
    axs[2].set_xlabel('x (mm)')
    axs[2].set_ylabel('Résidus')
    axs[2].set_title('Résidus bruts'); axs[2].grid(alpha=0.25)

    # P3 : Bootstrap V
    bV = result.boot_V
    if len(bV) > 2:
        axs[3].hist(bV, bins=20, color='coral',
                    edgecolor='white', alpha=0.85)
        axs[3].axvline(result.V_total, color='r', lw=2,
                       label=f'V={result.V_total:.5f}')
        ci_v = result.V_ci95
        axs[3].axvline(ci_v[0], color='gray', lw=1, ls=':')
        axs[3].axvline(ci_v[1], color='gray', lw=1, ls=':')
        axs[3].set_xlabel('V'); axs[3].set_title('Bootstrap — V')
        axs[3].legend(fontsize=8); axs[3].grid(alpha=0.25)

    # P4 : Résidus normalisés
    axs[4].axhline(0,  color='k', lw=0.8)
    axs[4].axhline( 2, color='r', lw=0.8, ls='--', alpha=0.6)
    axs[4].axhline(-2, color='r', lw=0.8, ls='--', alpha=0.6)
    axs[4].plot(xmm, resn, 'g.', ms=1.5, alpha=0.4)
    axs[4].set_xlabel('x (mm)')
    axs[4].set_ylabel('Résidus / √μ')
    axs[4].set_title(
        f'Résidus normalisés (χ²={result.chi2_red:.3f})',
        fontsize=10)
    axs[4].grid(alpha=0.25)

    # P5 : Composantes V
    labels = ['V_source','V_Lindblad','V_rel','V_total']
    vals   = [result.V_source, result.V_lindblad,
              result.V_relativiste, result.V_total]
    colors = ['steelblue','coral','mediumpurple','green']
    bars   = axs[5].bar(labels, vals, color=colors,
                         edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, vals):
        axs[5].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.012,
                    f'{val:.4f}', ha='center',
                    fontsize=9, fontweight='bold')
    axs[5].set_ylim(0, 1.20)
    axs[5].set_ylabel('Visibilité')
    axs[5].set_title('Composantes physiques de V')
    axs[5].grid(alpha=0.25, axis='y')

    plt.suptitle('DTQEM v4.2-fixed', fontsize=12,
                  fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    if outfile:
        plt.savefig(outfile, dpi=150, bbox_inches='tight')
        print(f'  Figure → {outfile}')
    plt.close()


# ══════════════════════════════════════════════════════════════════
# 9. SAUVEGARDE CSV
# ══════════════════════════════════════════════════════════════════
def save_csv(result: FitResult,
             phys: PhysicsParameters,
             d_true: Optional[float],
             outfile: Path):
    with open(outfile, 'w', newline='',
              encoding='utf-8') as f:
        w = csv.writer(f)
        for row in [
            ['# DTQEM v4.2-fixed-doubleslit'],
            ['# Date', time.strftime(
                '%Y-%m-%dT%H:%M:%SZ', time.gmtime())],
            ['# Python', sys.version.split()[0]],
            ['# NumPy',  np.__version__],
            ['# SciPy',  scipy.__version__],
            [],
            ['# gamma_phi',   phys.gamma_phi],
            ['# tau_c',       phys.tau_c],
            ['# gamma_tau',   phys.gamma_tau()],
            ['# delta_tau',   phys.delta_tau_relativistic()],
            ['# V_deco',      phys.V_decoherence()],
            [],
        ]:
            w.writerow(row)

        ci_d = result.d_ci95
        ci_V = result.V_ci95
        w.writerow(['param','value','std',
                    'ci95_low','ci95_high','unit'])
        w.writerows([
            ['d',       result.d*1e6, result.d_std*1e6,
             ci_d[0]*1e6, ci_d[1]*1e6, 'μm'],
            ['I0',      result.I0, result.I0_std, '','','coups'],
            ['phi',     result.phi,result.phi_std,'','','rad'],
            ['V_source',result.V_source,'','','',''],
            ['V_lind',  result.V_lindblad,'','','',''],
            ['V_rel',   result.V_relativiste,'','','',''],
            ['V_deco',  result.V_deco,'','','',''],
            ['V_total', result.V_total,result.V_std,
             ci_V[0],ci_V[1],''],
            ['gamma_tau',result.gamma_tau_val,'','','',''],
            ['delta_tau',result.delta_tau_val,'','','','s'],
        ])
        w.writerow([])
        w.writerows([
            ['chi2_red',  result.chi2_red],
            ['R2',        result.r2],
            ['AICc',      result.aicc],
            ['boot_rate', result.boot_rate],
            ['fit_time',  result.fit_time],
        ])
        if d_true is not None:
            err = abs(result.d - d_true)/d_true*100
            ci  = result.d_ci95
            cov = (ci[0] <= d_true <= ci[1]
                   if not np.isnan(ci[0]) else None)
            w.writerows([
                ['d_true_um', d_true*1e6],
                ['err_rel_%', err],
                ['ci_covers', cov],
            ])
    print(f'  CSV → {outfile}')


# ══════════════════════════════════════════════════════════════════
# 10. POINT D'ENTRÉE — AVEC TEST BRUIT NUL
# ══════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    np.random.seed(42)
    out = Path('output_dtqem_v42_fixed')
    out.mkdir(exist_ok=True)

    print('\n' + '='*64)
    print('DTQEM v4.2-fixed — Validation')
    print('='*64)
    print(f'NumPy : {np.__version__}')
    print(f'SciPy : {scipy.__version__}')
    print(f'μ_B   : {Const.MU_B:.6e} J/T')

    # ── Paramètres physiques
    phys = PhysicsParameters(
        lambda_m    = 532e-9,
        L_m         = 1.0,
        a_m         = 50e-6,
        L_source    = 0.5,
        source_size = 10e-6,
        mass_kg     = 9.109e-31,
        velocity    = 1e6,
        gamma_phi   = 1e10,
        tau_c       = 1e-15,   # [FIX 4] femtoseconde
        B0          = 50.0,
        B1          = 2.0,
        dB0         = 2.0,
        dB1         = 0.5,
        read_noise  = 5.0,
    )

    x       = np.linspace(-6e-3, 6e-3, 800)
    d_true  = 250e-6
    sb_true = 3e-3
    I0_true = 1500.0
    phi_true = 0.15

    # ── Test méthodes
    print('\n  ── Test méthodes PhysicsParameters ──')
    print(f'  tau_interaction()        = '
          f'{phys.tau_interaction():.4e} s')
    print(f'  gamma_lorentz()          = '
          f'{phys.gamma_lorentz():.8f}')
    print(f'  delta_tau_relativistic() = '
          f'{phys.delta_tau_relativistic():.4e} s')
    print(f'  gamma_tau()              = '
          f'{phys.gamma_tau():.6f}')
    print(f'  V_source(d_true)         = '
          f'{phys.V_source(d_true):.6f}')
    print(f'  V_decoherence()          = '
          f'{phys.V_decoherence():.6f}')
    print(f'  V_effective(d_true)      = '
          f'{phys.V_effective(d_true):.6f}')

    print(f'\n  d_true  = {d_true*1e6:.1f} μm')
    print(f'  γτ      = {phys.gamma_tau():.5f}')
    print(f'  Δτ      = {phys.delta_tau_relativistic():.3e} s')
    print(f'  V_pred  = {phys.V_effective(d_true):.5f}')

    # ══════════════════════════════════════════════════
    # [FIX 6] TEST SUR DONNÉES PROPRES (sans bruit)
    # ══════════════════════════════════════════════════
    print('\n' + '='*64)
    print('[FIX 6] TEST DONNÉES PROPRES (sans bruit)')
    print('='*64)

    I_clean = model_v42(x, I0_true, d_true, phi_true,
                         phys, sb_true)
    # Ajout fond uniquement
    I_clean_with_bg = I_clean  # B0/B1 déjà dans model_v42

    res_clean = fit_v42(
        x, I_clean, phys,
        sigma_b     = sb_true,
        fit_sigma_b = False,
        n_bootstrap = 10,   # minimal pour test rapide
        n_restarts  = 10,
        seed        = 42,
        verbose     = True,
    )

    err_clean = abs(res_clean.d - d_true) / d_true * 100
    print(f'\n  ── Résultat données propres ──')
    print(f'  d_true = {d_true*1e6:.4f} μm')
    print(f'  d_fit  = {res_clean.d*1e6:.4f} μm')
    print(f'  erreur = {err_clean:.4f}%'
          f'  {"✓ (<0.5%)" if err_clean < 0.5 else "✗ (>0.5%)"}')
    print(f'  χ²_red = {res_clean.chi2_red:.4f}')

    # ══════════════════════════════════════════════════
    # TEST AVEC BRUIT (Poisson + Gaussien)
    # ══════════════════════════════════════════════════
    print('\n' + '='*64)
    print('TEST DONNÉES BRUITÉES')
    print('='*64)

    I_noisy = (np.random.poisson(
                   np.maximum(I_clean, 1e-10)
               ).astype(float) +
               np.random.normal(0, phys.read_noise,
                                size=len(x)))
    I_noisy = np.maximum(I_noisy, 0.0)

    # [FIX 7] n_bootstrap=50
    result = fit_v42(
        x, I_noisy, phys,
        sigma_b     = sb_true,
        fit_sigma_b = False,
        n_bootstrap = 50,     # [FIX 7]
        n_restarts  = 15,
        seed        = 42,
        verbose     = True,
    )

    err = abs(result.d - d_true) / d_true * 100
    ci  = result.d_ci95
    print(f'\n  Erreur sur d : {err:.3f}%')
    print(f'  IC 95% couvre d_true : '
          f'{"✓" if ci[0]<=d_true<=ci[1] else "✗"}')

    plot_full(x, I_noisy, result, phys,
              d_true=d_true,
              title_suffix=' — données bruitées',
              outfile=out/'v42_fit_noisy.png')
    save_csv(result, phys, d_true,
             outfile=out/'v42_results_noisy.csv')

    # ══════════════════════════════════════════════════
    # [FIX 5] TEST sigma_b LIBRE
    # ══════════════════════════════════════════════════
    print('\n' + '='*64)
    print('[FIX 5] TEST sigma_b LIBRE (fit_sigma_b=True)')
    print('='*64)

    res_sb = fit_v42(
        x, I_noisy, phys,
        sigma_b     = 3e-3,   # point de départ
        fit_sigma_b = True,   # [FIX 5] libre
        n_bootstrap = 50,
        n_restarts  = 15,
        seed        = 42,
        verbose     = True,
    )
    print(f'\n  d_fit  = {res_sb.d*1e6:.4f} μm'
          f'  (vrai={d_true*1e6:.1f} μm)')
    print(f'  σ_b    = {res_sb.sigma_b*1e3:.3f} mm'
          f'  (vrai={sb_true*1e3:.1f} mm)')
    print(f'  erreur d = '
          f'{abs(res_sb.d-d_true)/d_true*100:.3f}%')

    # ══════════════════════════════════════════════════
    # Résumé final
    # ══════════════════════════════════════════════════
    print('\n' + '═'*64)
    print('Résumé comparatif')
    print('─'*64)
    print(f'{"Test":<30} {"d_fit (μm)":>12}'
          f' {"err%":>8} {"χ²_red":>8}')
    print('─'*64)
    tests = [
        ('Données propres',     res_clean),
        ('Données bruitées',    result),
        ('σ_b libre',           res_sb),
    ]
    for name, r in tests:
        err_r = abs(r.d - d_true)/d_true*100
        print(f'{name:<30} {r.d*1e6:>12.4f}'
              f' {err_r:>7.3f}% {r.chi2_red:>8.4f}')
    print('═'*64)

    print('\nFichiers générés :')
    for fp in sorted(out.iterdir()):
        print(f'  {fp.name:<48}'
              f'{fp.stat().st_size/1024:6.1f} KB')
    print('\n✅  DTQEM v4.2-fixed — Terminé.')


DTQEM v4.2-fixed — Validation
NumPy : 2.0.2
SciPy : 1.16.3
μ_B   : 9.274010e-24 J/T

  ── Test méthodes PhysicsParameters ──
  tau_interaction()        = 5.0000e-11 s
  gamma_lorentz()          = 1.00000556
  delta_tau_relativistic() = 2.7816e-16 s
  gamma_tau()              = 0.500000
  V_source(d_true)         = 0.999855
  V_decoherence()          = 0.459249
  V_effective(d_true)      = 0.459182

  d_true  = 250.0 μm
  γτ      = 0.50000
  Δτ      = 2.782e-16 s
  V_pred  = 0.45918

[FIX 6] TEST DONNÉES PROPRES (sans bruit)

════════════════════════════════════════════════════════════════
DTQEM v4.2-fixed — Ajustement
════════════════════════════════════════════════════════════════
  ════════════════════════════════════════════════════
    PhysicsParameters
  ────────────────────────────────────────────────────
    λ             = 532.0000 nm
    L             = 1.0000 m
    a             = 50.0000 μm
    L_source      = 0.5000 m
    source_size   = 10.000 μm
    velocity      = 1.000